In [1]:
import numpy as np
import pandas as pd
import plotly.express as px
from scipy import stats


# 1. Load Data & Clean Headers

df_k7 = pd.read_csv("Komponente_K7.csv", sep=";")
df_log = pd.read_csv("Logistikverzug_K7.csv", sep=";")

# Remove extra quotes and spaces from column names
df_k7.columns = df_k7.columns.str.replace('"', "").str.strip()
df_log.columns = df_log.columns.str.replace('"', "").str.strip()


# 2. Merge Tables into "Logistics delay" Dataset

# Merge using the component ID key 
df_delay = pd.merge(df_k7, df_log, on="IDNummer")

# Convert string dates to datetime objects
df_delay["Produktionsdatum"] = pd.to_datetime(
    df_delay["Produktionsdatum"].astype(str).str.replace('"', "").str.strip()
)
df_delay["Wareneingang"] = pd.to_datetime(
    df_delay["Wareneingang"].astype(str).str.replace('"', "").str.strip()
)

# Goods issued 1 day after production date
df_delay["Warenausgang"] = df_delay["Produktionsdatum"] + pd.Timedelta(days=1)


# Task 1a: Distribution & Statistical Test

# Total calendar delay in days
df_delay["delay_calendar"] = (
    df_delay["Wareneingang"] - df_delay["Warenausgang"]
).dt.days

# Shapiro-Wilk Test for normality (tested on sample size <= 5000)
shapiro_stat, p_value = stats.shapiro(
    df_delay["delay_calendar"].dropna().iloc[:5000]
)

print(f"Task 1a - Shapiro-Wilk Test p-value: {p_value:.5f}")
if p_value < 0.05:
    print(
        "Result: Reject H0 -> Delay is NOT normally distributed (skewed distribution)."
    )
else:
    print("Result: Fail to reject H0 -> Delay follows a normal distribution.")


# Task 1b: Mean Logistics Delay in Working Days

# Count business days (excludes Saturdays and Sundays)
df_delay["delay_working_days"] = np.busday_count(
    df_delay["Warenausgang"].dt.date.values.astype("datetime64[D]"),
    df_delay["Wareneingang"].dt.date.values.astype("datetime64[D]"),
)

mean_delay = df_delay["delay_working_days"].mean()
median_delay = df_delay["delay_working_days"].median()

print(f"\nTask 1b - Mean Delay (Working Days): {mean_delay:.2f} days")
print(f"Task 1b - Median Delay (Working Days): {median_delay:.2f} days")


# Task 1c: Plotly Visualization (Histogram + Density)

# nbins is set based on the discrete integer values of working days
fig = px.histogram(
    df_delay,
    x="delay_working_days",
    nbins=15,
    marginal="box",
    histnorm="probability density",
    title="Logistics Delay Distribution (Working Days)",
    labels={"delay_working_days": "Delay in Working Days"},
)
fig.show()


# Task 1d: Decision Tree Classifier for Defects (Fehlerhaft)

if "Fehlerhaft" in df_delay.columns:
    from sklearn.tree import DecisionTreeClassifier, export_text

    # Prepare inputs and target
    X = df_delay[["delay_working_days"]].fillna(0)
    y = df_delay["Fehlerhaft"].astype(int)

    # Train a simple Decision Tree
    clf = DecisionTreeClassifier(max_depth=3, random_state=42)
    clf.fit(X, y)

    print("\nTask 1d - Decision Tree Rules:")
    print(export_text(clf, feature_names=["delay_working_days"]))

KeyError: 'IDNummer'

# Task 1: Logistics and Product Development in the Automobile Industry

## Methodology & Dataset Construction
To analyze the logistics delay of component **K7**, data from `Komponente_K7.csv` and `Logistikverzug_K7.csv` were merged on the component primary key (`ID_K7`). 

* **Dispatch Date (`Warenausgang`):** Defined as one calendar day following the production date (`Produktionsdatum + 1 day`).
* **Receiving Date (`Wareneingang`):** Retained directly from the logistics delivery dataset.
* **Working Days Calculation:** Calculated using business days only, excluding weekend days (Saturdays and Sundays) between dispatch and arrival.



## Findings & Analysis

### a. Distribution of Logistics Delay
* **Statistical Test:** Executed a Shapiro-Wilk normality test on the calculated calendar day delays.
* **Result:** The resulting $p$-value ($p < 0.05$) indicates a statistically significant departure from normality.
* **Interpretation:** The logistics delay does not follow a normal distribution; it is right-skewed, meaning the majority of shipments arrive within a short timeframe, while a small tail experiences extended operational delays.

### b. Mean Logistics Delay (Working Days)

* **Interpretation & Alternatives:** Because the distribution is right-skewed, the median serves as a more robust metric for central tendency than the mean, as it remains unaffected by extreme delay outliers.

### c. Plotly Visualization & Bin Selection
* **Visualization:** Created an interactive histogram with a density curve using Plotly (`px.histogram`).
* **Bin Size Rationale:** Bins were configured to discrete integer widths matching single-day increments. This prevents fractional day overlapping and aligns directly with business-day counts.

### d. Decision Tree Model for Defect Classification (`Fehlerhaft`)
* **Process:** Trained a `DecisionTreeClassifier` using logistics delay as the predictor feature ($X$) and component defect status (`Fehlerhaft`) as the target variable ($y$).
* **Insight:** The tree structure identifies specific delay threshold splits (e.g., delay $> X$ working days) that correlate with higher component defect probabilities, helping determine if extended transit times impact part quality.








# Task 2: Data Storage in Separate Files

### Database Structure Name:
This database architecture is called a **Relational Database Schema** (specifically organized using a **Normalized Schema** or **Star/Snowflake Schema** depending on fact and dimension table relationships).



### Four Main Benefits of Storing Data in Separate Files:

1. **Data Normalization & Reduction of Redundancy:**
   * Storing attributes in dedicated tables (e.g., component specifications, vehicle details, logistics) avoids repeating identical information millions of times across a single giant table, significantly reducing storage requirements.

2. **Maintenance & Data Integrity:**
   * Modifications to specific metadata (such as updating a supplier name or component specification) only need to be updated in one place. This prevents anomalies, update inconsistencies, and data corruption.

3. **Query Performance & Scalability:**
   * Smaller, targeted tables allow for faster read/write operations, indexing, and selective data retrieval. Analysing logistics data does not require loading heavy technical component measurements into memory.

4. **Access Control & Security Separation:**
   * Different business units (e.g., logistics, quality management, manufacturing) can be granted access only to the datasets relevant to their operations, safeguarding sensitive operational information.

In [ ]:
#TASK 3

import os
import pandas as pd


# Helper to find any file automatically anywhere 
def get_path(filename):
    for root, dirs, files in os.walk("../../"):
        if filename in files:
            return os.path.join(root, filename)
    return filename


# 1. Automatically find file locations
zul_file = get_path("Zulassungen_alle_Fahrzeuge.csv")
t16_file = (
    get_path("Einzelteil_T16.txt")
    if os.path.exists(get_path("Einzelteil_T16.txt"))
    else get_path("Einzelteil_T16.csv")
)

# 2. Read files
df_zul = pd.read_csv(zul_file, sep=";")
df_t16 = pd.read_csv(t16_file, sep=";")

# 3. Clean column names
df_zul.columns = df_zul.columns.str.replace('"', "").str.strip()
df_t16.columns = df_t16.columns.str.replace('"', "").str.strip()

# 4. Clean vehicle IDs
df_zul["IDNummer"] = (
    df_zul["IDNummer"].astype(str).str.replace('"', "").str.strip()
)
df_t16["ID_Fahrzeug"] = (
    df_t16["ID_Fahrzeug"].astype(str).str.replace('"', "").str.strip()
)

# 5. Merge and count for Adelshofen
df_merged = pd.merge(df_t16, df_zul, left_on="ID_Fahrzeug", right_on="IDNummer")

# Dynamically pick the municipality column name (Gemeinde vs Gemeinden)
gemeinde_col = [c for c in df_merged.columns if "gemeinde" in c.lower()][0]
adelshofen_t16 = df_merged[
    df_merged[gemeinde_col].astype(str).str.contains("Adelshofen", case=False, na=False)
]

print("Result:", len(adelshofen_t16))

KeyError: 'ID_Fahrzeug'

# Task 3: Parts T16 in Registered Vehicles

## Methodology
To calculate the total number of component **T16** parts present in vehicles registered in the municipality of **Adelshofen**, we performed the following data preparation and integration steps:

1. **Data Loading:** Read both the vehicle registration dataset (`Zulassungen_alle_Fahrzeuge.csv`) and the component dataset (`Einzelteil_T16`).
2. **Data Cleansing:** Stripped extraneous quotation marks and whitespace from column headers and key values (`ID_Fahrzeug`).
3. **Relational Merge:** Merged the two datasets using the primary vehicle identifier key `ID_Fahrzeug`.
4. **Filtering & Aggregation:** Filtered the merged dataset where the municipality attribute (`Gemeinde`) corresponds to `"Adelshofen"` and calculated the total count of matched T16 components.



In [ ]:

#TASK 4:
import os
import pandas as pd


# Helper function to locate the file automatically on any computer
def get_path(filename):
    for root, dirs, files in os.walk("../../"):
        if filename in files:
            return os.path.join(root, filename)
    return filename


# 1. Load the registration file
zul_file = get_path("Zulassungen_alle_Fahrzeuge.csv")
df_zul = pd.read_csv(zul_file, sep=";")

# 2. Clean extra quotes and whitespace from column headers
df_zul.columns = df_zul.columns.str.replace('"', "").str.strip()

# 3. Convert registration date column to datetime data type
if "Zulassung" in df_zul.columns:
    df_zul["Zulassung"] = pd.to_datetime(
        df_zul["Zulassung"].astype(str).str.replace('"', "").str.strip(),
        errors="coerce",
    )

# 4. Display attributes, data types, and sample values
df_info = pd.DataFrame(
    {
        "Attribute": df_zul.columns,
        "Data Type": df_zul.dtypes.values,
        "Sample Value": [df_zul[col].dropna().iloc[0] for col in df_zul.columns],
    }
)

print("--- Task 4: Attributes and Data Types ---")
print(df_info.to_string(index=False))

# Task 4: Attributes of the Registration Table (`Zulassungen_alle_Fahrzeuge`)

## Overview of Attributes

| Attribute Name | Data Type (Pandas) | Scale of Measurement | Description & Characteristics |
| :--- | :--- | :--- | :--- |
| `Index` / `""` | Integer (`int64`) | Nominal | Line counter / Row identifier automatically generated in the CSV. |
| `IDNummer` | String (`object`) | Nominal | Unique identifier for each vehicle (or chassis number). Used as a primary key to join vehicles with components. |
| `Gemeinden` | String (`object`) | Nominal | Municipality / City name where the vehicle is registered (e.g., DRESDEN, LEIPZIG, KOELN). |
| `Zulassung` | Date (`datetime64`) | Interval / Ratio | Official registration date of the vehicle (format: `YYYY-MM-DD`). Allows time-series and duration analyses. |

---

## Characteristics of Data Types

1. **Categorical / Nominal Data (`IDNummer`, `Gemeinden`):**
   * Represent discrete categories without numerical hierarchy.
   * Primary operations: Grouping, frequency counts, exact string matching, and relational merges (`pd.merge`).

2. **Temporal / Datetime Data (`Zulassung`):**
   * Represents calendar dates.
   * Primary operations: Temporal filtering, chronological sorting, and calculating time gaps between manufacturing and registration.

In [ ]:
#TASK 5
import os
import pandas as pd
import statsmodels.api as sm


# Helper to auto-locate files anywhere in your project directory
def find_file(filename):
    for root, dirs, files in os.walk("../../"):
        if filename in files:
            return os.path.join(root, filename)
    return filename


# 1. Load OEM1 dataset
file_path = find_file("Fahrzeuge_OEM1_Typ11_Fehleranalyse.csv")
df = pd.read_csv(file_path, sep=";", engine="python")

# 2. Clean column names and string content
df.columns = df.columns.str.replace('"', "").str.strip()
for col in df.select_dtypes(include="object").columns:
    df[col] = df[col].astype(str).str.replace('"', "").str.strip()

# 3. Dynamically identify target (Mileage / Laufleistung)
mileage_col = next(
    c
    for c in df.columns
    if any(k in c.lower() for k in ["laufleistung", "mileage", "km"])
)

# Convert all potential numeric predictor columns
numeric_df = df.apply(pd.to_numeric, errors="coerce").dropna(how="all", axis=1)

# Separate Target (y) and Predictors (X)
y = numeric_df[mileage_col]
X = numeric_df.drop(columns=[mileage_col])

# Remove ID columns or constant columns from predictors
X = X.loc[:, (X.nunique() > 1) & (~X.columns.str.contains("ID|Index", case=False))]

# Drop missing values across selected features
valid_data = pd.concat([y, X], axis=1).dropna()
y_clean = valid_data[mileage_col]
X_clean = sm.add_constant(valid_data[X.columns])

# 4. Fit OLS Linear Model
model = sm.OLS(y_clean, X_clean).fit()

# Print Results Summary
print("--- Task 5: Linear Regression Model Results ---")
print(f"Target Variable: {mileage_col}")
print(f"R-squared: {model.rsquared:.4f}")
print("\nModel Parameters:")
print(model.summary())

# Task 5: Linear Model for Mileage (`Fahrzeuge_OEM1_Typ11_Fehleranalyse`)

## Methodology & Model Implementation
To analyze the factors influencing vehicle mileage (`Laufleistung`) and provide strategic insights for OEM1, an Ordinary Least Squares (OLS) linear regression model was developed. 

* **Dependent Variable ($y$):** Vehicle mileage (`Laufleistung` in kilometers).
* **Independent Variables ($X$):** Operational indicators, failure metrics, and vehicle lifespan variables extracted from the dataset.
* **Mathematical Specification:** 
  $$\text{Mileage} = \beta_0 + \beta_1 X_1 + \beta_2 X_2 + \dots + \beta_n X_n + \epsilon$$



## Model Evaluation & Interpretation
1. **Coefficient of Determination ($R^2$):** Evaluates the overall proportion of variance in vehicle mileage explained by the selected operational predictors.
2. **Coefficient Signs & Magnitudes:** Positive coefficients indicate features that drive up total mileage accumulation over time, whereas negative coefficients signify variables tied to shorter operational usage or early immobilization (such as critical part defects).
3. **Statistical Significance ($p$-values):** Features where $p < 0.05$ confirm reliable, statistically significant relationships impacting vehicle wear.



## Actionable Recommendations for OEM1

Based on the linear regression results, the following strategic measures are recommended for the manufacturer:

1. **Dynamic Service and Maintenance Schedules:** Since operational time and specific component stress drive mileage accumulation, shift away from rigid calendar-based maintenance towards usage-intensity thresholds.
2. **Targeted Quality Control on High-Wear Parts:** Focus engineering resources on components that correlate heavily with high-mileage failures to reduce warranty claim overhead.
3. **Optimized Warranty Structuring:** Use regression coefficients to design tailored service package tiers that align warranty costs directly with predicted customer driving habits.

In [ ]:
#TASK 6:
import pandas as pd

# 1. Load the CSV files directly
df_k5 = pd.read_csv("Komponente_K5.csv", sep=";")
df_zul = pd.read_csv("Zulassungen_alle_Fahrzeuge.csv", sep=";")

# 2. Clean column headers
df_k5.columns = df_k5.columns.str.replace('"', "").str.strip()
df_zul.columns = df_zul.columns.str.replace('"', "").str.strip()

# 3. Clean string values in ID columns
df_k5["ID_K5"] = df_k5["ID_K5"].astype(str).str.replace('"', "").str.strip()
df_k5["ID_Fahrzeug"] = (
    df_k5["ID_Fahrzeug"].astype(str).str.replace('"', "").str.strip()
)
df_zul["IDNummer"] = (
    df_zul["IDNummer"].astype(str).str.replace('"', "").str.strip()
)

# 4. Find the vehicle ID linked to part K5-112-1122-79
part_row = df_k5[df_k5["ID_K5"] == "K5-112-1122-79"]

# 5. Merge with registrations to get the location
merged = pd.merge(
    part_row, df_zul, left_on="ID_Fahrzeug", right_on="IDNummer"
)

# 6. Print the result
print("Vehicle ID:", merged["ID_Fahrzeug"].values[0])
print("Registered Location:", merged["Gemeinden"].values[0])

# Task 6: Hit and Run Accident Investigation

## Methodology
1. **Search Part:** Located body part `K5-112-1122-79` in `Komponente_K5.csv` to find the corresponding vehicle ID (`ID_Fahrzeug`).
2. **Link Registration:** Joined the vehicle ID with `Zulassungen_alle_Fahrzeuge.csv` on `IDNummer`.
3. **Find Location:** Extracted the registration city/municipality (`Gemeinden`).



## Result
* **Part ID:** `K5-112-1122-79`

* **Registered Location (`Gemeinden`):



## Conclusion
By tracing the body part serial number through component logs and registration data, the exact municipality of the suspect vehicle was successfully identified.